# P6 Sectoral Drivers of the UK Emissions Gap

**Purpose.** This notebook produces a locally reproducible first version of the P6 analysis:

- clean DESNZ EEP 2024-2050 sectoral projection data;
- cover the major TES sectors: transport, buildings and product uses, electricity supply, industry, agriculture, fuel supply, waste, LULUCF and IAS;
- rank 2050 residual emissions;
- identify sectoral changes from the latest historical year to 2050;
- generate figures and tables for the Results section;
- optionally compare DESNZ sectors with the cautious CCC sector-alignment table prepared in P5.

**Main source.** DESNZ Energy and Emissions Projections 2024-2050, Annex A: GHG by TES sector. The `Reference` sheet is used as the main current-policy baseline. Other EEP scenarios can be added later as sensitivity checks.

**?????** ?? Notebook ? P6 ??????? DESNZ sectoral projections ??????????? 2050 ???????sector projection ?????? Results ??????????

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 240,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

## 1. Locate project folders

Run this notebook from anywhere inside the dissertation project. It searches upward and also checks the known project path.

In [ ]:
def find_project_root(start=None):
    candidates = []
    if start is not None:
        p = Path(start).resolve()
    else:
        p = Path.cwd().resolve()
    candidates.extend([p, *p.parents])
    candidates.append(Path(r"E:\UCL Final Essay"))
    for candidate in candidates:
        if (candidate / "Data_raw").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing Data_raw. Please run inside the dissertation project folder.")

PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "Data_raw"
P6_ROOT = PROJECT_ROOT / "p6_sector_analysis"
INTERMEDIATE_DIR = P6_ROOT / "data_intermediate"
PROCESSED_DIR = P6_ROOT / "data_processed"
TABLES_DIR = P6_ROOT / "tables"
FIGURES_DIR = P6_ROOT / "figures"

for folder in [P6_ROOT, INTERMEDIATE_DIR, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("P6_ROOT:", P6_ROOT)

## 2. Locate and prepare DESNZ sector annex files

The original DESNZ annex files are `.ods`. This notebook prefers already-converted `.xlsx` copies, because they are easier to read locally with `pandas` + `openpyxl`. If the `.xlsx` copies are missing, it will try to convert the `.ods` files using LibreOffice/`soffice`.

In [ ]:
def find_first(pattern, root=DATA_RAW, must_contain=None):
    matches = list(root.rglob(pattern))
    if must_contain:
        matches = [p for p in matches if must_contain.lower() in str(p).lower()]
    if not matches:
        return None
    # Prefer the full DESNZ EEP annex folder when available.
    matches = sorted(matches, key=lambda p: ("DESNZ Energy and Emissions Projections" not in str(p), len(str(p))))
    return matches[0]

def find_soffice():
    candidates = [
        shutil.which("soffice"),
        r"C:\Program Files\LibreOffice\program\soffice.exe",
        r"C:\Program Files (x86)\LibreOffice\program\soffice.exe",
    ]
    for c in candidates:
        if c and Path(c).exists():
            return str(c)
    return None

def convert_ods_to_xlsx(ods_path, out_dir=INTERMEDIATE_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    xlsx_path = out_dir / (Path(ods_path).stem + ".xlsx")
    if xlsx_path.exists():
        return xlsx_path
    soffice = find_soffice()
    if soffice is None:
        raise FileNotFoundError(
            "LibreOffice/soffice was not found. Either install LibreOffice or manually save the ODS as XLSX."
        )
    cmd = [
        soffice,
        "--headless",
        "--norestore",
        "--convert-to",
        "xlsx",
        "--outdir",
        str(out_dir),
        str(ods_path),
    ]
    completed = subprocess.run(cmd, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError("LibreOffice conversion failed:\n" + completed.stderr)
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Expected converted file not found: {xlsx_path}")
    return xlsx_path

TES_XLSX = INTERMEDIATE_DIR / "Annex_A_GHG_by_TES_sector.xlsx"
NZS_XLSX = INTERMEDIATE_DIR / "Annex_A_GHG_by_NZS_category.xlsx"

if not TES_XLSX.exists():
    tes_ods = find_first("Annex_A_GHG_by_TES_sector.ods", must_contain="DESNZ Energy and Emissions Projections")
    if tes_ods is None:
        raise FileNotFoundError("Could not find Annex_A_GHG_by_TES_sector.ods in Data_raw.")
    TES_XLSX = convert_ods_to_xlsx(tes_ods)

if not NZS_XLSX.exists():
    nzs_ods = find_first("Annex_A_GHG_by_NZS_category.ods", must_contain="DESNZ Energy and Emissions Projections")
    if nzs_ods is not None:
        NZS_XLSX = convert_ods_to_xlsx(nzs_ods)

print("TES_XLSX:", TES_XLSX)
print("NZS_XLSX:", NZS_XLSX if NZS_XLSX.exists() else "not available / not needed for first pass")

## 3. Read DESNZ TES Reference scenario

The `Reference` sheet is the main DESNZ EEP current-policy projection. We keep the broad TES sector rows and remove accounting-total rows.

In [ ]:
TES_SCENARIO = "Reference"
raw = pd.read_excel(TES_XLSX, sheet_name=TES_SCENARIO, header=2)

# Remove empty rows and standardise columns.
raw = raw.dropna(how="all").copy()
raw.columns = [str(c).strip() for c in raw.columns]

year_cols = [str(y) for y in range(1990, 2051) if str(y) in raw.columns]
period_cols = [c for c in ["CB4", "CB5", "CB6", "CB7"] if c in raw.columns]

all_ghg = raw[(raw["GHG"] == "GHG (All)") & (raw["units"] == "MtCO2e")].copy()
all_ghg["coverage"] = all_ghg["coverage"].astype(str).str.strip()

accounting_total_mask = all_ghg["coverage"].str.contains("Total emissions|Net Carbon Account", case=False, regex=True, na=False)
sector_wide = all_ghg[~accounting_total_mask].copy()

for c in year_cols + period_cols:
    sector_wide[c] = pd.to_numeric(sector_wide[c], errors="coerce")

expected_sectors = [
    "Agriculture",
    "Buildings and product uses",
    "Domestic Transport",
    "Electricity supply",
    "Fuel supply",
    "IAS",
    "Industry",
    "LULUCF",
    "Waste",
]
sector_wide = sector_wide[sector_wide["coverage"].isin(expected_sectors)].copy()
sector_wide = sector_wide.sort_values("coverage").reset_index(drop=True)

sector_wide[["coverage", "2023", "2030", "2035", "2050", "CB6"]]

## 4. Build clean long-format sector dataset

This is the main reusable P6 processed dataset: one row per sector per year.

In [ ]:
sector_long = sector_wide.melt(
    id_vars=["coverage"],
    value_vars=year_cols,
    var_name="year",
    value_name="emissions_MtCO2e",
)
sector_long["year"] = sector_long["year"].astype(int)
sector_long["scenario"] = TES_SCENARIO
sector_long = sector_long.rename(columns={"coverage": "tes_sector"})
sector_long = sector_long[["scenario", "tes_sector", "year", "emissions_MtCO2e"]].sort_values(["tes_sector", "year"])

sector_long_path = PROCESSED_DIR / "p6_desnz_tes_reference_sector_long.csv"
sector_long.to_csv(sector_long_path, index=False)

print("Saved:", sector_long_path)
display(sector_long.head())

## 5. 2050 residual emissions ranking

This table answers the first P6 question: which sectors contribute most to remaining projected emissions in 2050?

In [ ]:
key_years = ["2023", "2030", "2035", "2050"]
ranking = sector_wide[["coverage", *key_years, "CB6"]].copy()
ranking = ranking.rename(columns={"coverage": "tes_sector"})

inc_total_2050 = float(all_ghg.loc[all_ghg["coverage"].eq("Total emissions (inc. IAS)"), "2050"].iloc[0])
exc_total_2050 = float(all_ghg.loc[all_ghg["coverage"].eq("Total emissions (exc. IAS)"), "2050"].iloc[0])

ranking["change_2023_2050_MtCO2e"] = ranking["2050"] - ranking["2023"]
ranking["share_of_2050_inc_IAS_total_pct"] = ranking["2050"] / inc_total_2050 * 100
ranking["rank_2050_residual"] = ranking["2050"].rank(method="first", ascending=False).astype(int)
ranking = ranking.sort_values("2050", ascending=False).reset_index(drop=True)

ranking_path = TABLES_DIR / "p6_desnz_2050_residual_emissions_ranking.csv"
ranking.to_csv(ranking_path, index=False)

print("DESNZ 2050 total inc IAS:", round(inc_total_2050, 1), "MtCO2e")
print("DESNZ 2050 total exc IAS:", round(exc_total_2050, 1), "MtCO2e")
print("Saved:", ranking_path)
display(ranking)

## 6. Sectoral contribution to change, 2023-2050

This table separates sectors where projected emissions fall from sectors where residual emissions remain high or increase.

In [ ]:
change_table = ranking[[
    "tes_sector", "2023", "2050", "change_2023_2050_MtCO2e", "share_of_2050_inc_IAS_total_pct", "rank_2050_residual"
]].copy()
change_table["interpretation_flag"] = np.where(
    change_table["change_2023_2050_MtCO2e"] > 0,
    "Projected increase from 2023 to 2050",
    "Projected reduction from 2023 to 2050"
)

change_path = TABLES_DIR / "p6_sector_change_2023_to_2050.csv"
change_table.to_csv(change_path, index=False)
print("Saved:", change_path)
display(change_table)

## 7. Figures

The figures below are intended for the P6 Results subsection.

In [ ]:
# Figure 1: sectoral projections for all major sectors.
fig, ax = plt.subplots(figsize=(9.0, 5.4))
plot_df = sector_long[(sector_long["year"] >= 2023) & (sector_long["year"] <= 2050)].copy()

# Put high-priority sectors first in the legend.
sector_order = ranking["tes_sector"].tolist()
for sector in sector_order:
    s = plot_df[plot_df["tes_sector"] == sector]
    ax.plot(s["year"], s["emissions_MtCO2e"], linewidth=2.0, label=sector)

ax.set_title("DESNZ EEP Reference sectoral emissions projections, 2023-2050")
ax.set_xlabel("Year")
ax.set_ylabel("MtCO2e")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, frameon=False, loc="upper right")
fig.tight_layout()
fig1_path = FIGURES_DIR / "p6_desnz_sector_projection_2023_2050.png"
fig.savefig(fig1_path, bbox_inches="tight")
plt.show()
print("Saved:", fig1_path)

In [ ]:
# Figure 2: 2050 residual emissions ranking.
fig, ax = plt.subplots(figsize=(8.2, 5.0))
bar_df = ranking.sort_values("2050", ascending=True)
colors = ["#7A8A99" if s not in ["Buildings and product uses", "Domestic Transport", "Electricity supply"] else "#0B6E8A" for s in bar_df["tes_sector"]]
ax.barh(bar_df["tes_sector"], bar_df["2050"], color=colors)
ax.set_title("DESNZ projected residual emissions by sector in 2050")
ax.set_xlabel("MtCO2e in 2050")
ax.grid(axis="x", alpha=0.25)
for y, v in enumerate(bar_df["2050"]):
    ax.text(v + 1, y, f"{v:.1f}", va="center", fontsize=8)
fig.tight_layout()
fig2_path = FIGURES_DIR / "p6_desnz_2050_residual_emissions_ranking.png"
fig.savefig(fig2_path, bbox_inches="tight")
plt.show()
print("Saved:", fig2_path)

In [ ]:
# Figure 3: change from 2023 to 2050.
fig, ax = plt.subplots(figsize=(8.2, 5.0))
change_plot = change_table.sort_values("change_2023_2050_MtCO2e")
colors = np.where(change_plot["change_2023_2050_MtCO2e"] >= 0, "#B23A48", "#2E7D60")
ax.barh(change_plot["tes_sector"], change_plot["change_2023_2050_MtCO2e"], color=colors)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set_title("Projected sectoral emissions change, 2023-2050")
ax.set_xlabel("Change in MtCO2e; positive means higher emissions in 2050")
ax.grid(axis="x", alpha=0.25)
for y, v in enumerate(change_plot["change_2023_2050_MtCO2e"]):
    if v < 0:
        ax.text(v + 1.0, y, f"{v:.1f}", va="center", ha="left", fontsize=8, color="white" if abs(v) > 10 else "#111111")
    else:
        ax.text(v + 0.5, y, f"{v:.1f}", va="center", ha="left", fontsize=8, color="#111111")
fig.tight_layout()
fig3_path = FIGURES_DIR / "p6_sector_change_2023_to_2050.png"
fig.savefig(fig3_path, bbox_inches="tight")
plt.show()
print("Saved:", fig3_path)

## 8. Optional: cautious DESNZ-CCC sector comparison

This is deliberately labelled cautious because the P5 alignment table shows that several DESNZ and CCC sectors do not match one-to-one. Use this only to support interpretation, not as the main P6 headline.

In [ ]:
alignment_path = PROJECT_ROOT / "p4_p5_local_reproduction" / "tables" / "p5_ccc_desnz_sector_alignment.csv"
if alignment_path.exists():
    alignment = pd.read_csv(alignment_path)
    alignment["gap_DESNZ_minus_CCC_2050_MtCO2e"] = alignment["desnz_2050_MtCO2e"] - alignment["ccc7_2050_MtCO2e"]
    alignment_out = TABLES_DIR / "p6_cautious_desnz_ccc_sector_alignment_2050.csv"
    alignment.to_csv(alignment_out, index=False)
    print("Saved:", alignment_out)
    display(alignment.sort_values("gap_DESNZ_minus_CCC_2050_MtCO2e", ascending=False))
else:
    print("P5 sector alignment table not found. Skipping cautious CCC comparison.")

In [ ]:
if alignment_path.exists():
    plot_align = alignment.sort_values("gap_DESNZ_minus_CCC_2050_MtCO2e", ascending=True)
    fig, ax = plt.subplots(figsize=(8.6, 5.2))
    ax.barh(plot_align["desnz_tes_sector"], plot_align["gap_DESNZ_minus_CCC_2050_MtCO2e"], color="#8A3FFC")
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_title("Cautious broad-sector gap: DESNZ 2050 minus CCC7 2050")
    ax.set_xlabel("MtCO2e; broad alignment only")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig4_path = FIGURES_DIR / "p6_cautious_desnz_ccc_sector_gap_2050.png"
    fig.savefig(fig4_path, bbox_inches="tight")
    plt.show()
    print("Saved:", fig4_path)

## 9. Quality checks

These checks are designed to catch obvious extraction mistakes before using the figures in the dissertation.

In [ ]:
checks = []

available_sectors = set(sector_wide["coverage"])
missing_sectors = sorted(set(expected_sectors) - available_sectors)
checks.append({
    "check": "Expected TES sectors are present",
    "status": "PASS" if not missing_sectors else "FAIL",
    "details": "; ".join(missing_sectors) if missing_sectors else "all expected sectors found",
})

required_years = set(str(y) for y in range(2023, 2051))
missing_years = sorted(required_years - set(year_cols))
checks.append({
    "check": "Projection years 2023-2050 are present",
    "status": "PASS" if not missing_years else "FAIL",
    "details": "; ".join(missing_years) if missing_years else "all years 2023-2050 found",
})

sector_sum_2050 = float(sector_wide["2050"].sum())
checks.append({
    "check": "Sector sum approximately equals total emissions inc IAS in 2050",
    "status": "PASS" if abs(sector_sum_2050 - inc_total_2050) < 0.2 else "WARN",
    "details": f"sector_sum={sector_sum_2050:.3f}; inc_total={inc_total_2050:.3f}; diff={sector_sum_2050-inc_total_2050:.3f}",
})

checks.append({
    "check": "Main output tables saved",
    "status": "PASS" if sector_long_path.exists() and ranking_path.exists() and change_path.exists() else "FAIL",
    "details": "sector_long, residual ranking, and change table",
})

checks.append({
    "check": "Main figures saved",
    "status": "PASS" if fig1_path.exists() and fig2_path.exists() and fig3_path.exists() else "FAIL",
    "details": "projection, residual ranking, and change figures",
})

qc = pd.DataFrame(checks)
qc_path = TABLES_DIR / "p6_sector_analysis_quality_checks.csv"
qc.to_csv(qc_path, index=False)
print("Saved:", qc_path)
display(qc)

## 10. Draft interpretation notes for P6 Results

These notes are not final prose. They are a safe starting point for the P6 Results subsection.

In [ ]:
top3 = ranking.head(3)
notes = []
notes.append("P6 headline: under the DESNZ EEP Reference projection, residual emissions in 2050 are concentrated in a small number of sectors rather than spread evenly across the economy.")
notes.append("The largest projected 2050 residual sector is " + top3.iloc[0]["tes_sector"] + f" ({top3.iloc[0]['2050']:.1f} MtCO2e).")
next_sector_text = ", ".join([f"{row['tes_sector']} ({row['2050']:.1f} MtCO2e)" for _, row in top3.iloc[1:].iterrows()])
notes.append("The next largest sectors are " + next_sector_text + ".")
notes.append("Transport remains important because it starts from a high 2023 level, even though the DESNZ projection shows substantial reductions by 2050.")
notes.append("Buildings and product uses require careful interpretation because this DESNZ category combines several sources and is not a clean one-to-one match with CCC residential and non-residential buildings.")
notes.append("Electricity supply should be discussed as a system-enabling sector: it matters directly through residual supply emissions and indirectly through transport/building electrification.")
notes.append("CCC sector comparison should remain cautious unless sector boundaries are fully bridged.")

notes_path = TABLES_DIR / "p6_results_interpretation_notes.txt"
notes_path.write_text("\n".join(f"- {n}" for n in notes), encoding="utf-8")
print("Saved:", notes_path)
print("\n".join(f"- {n}" for n in notes))

## Next step

Use the generated tables and figures to draft the P6 Results subsection:

1. Start with the 2050 residual ranking.
2. Explain transport, buildings and electricity in more detail.
3. Use industry, agriculture, fuel supply, waste, LULUCF and IAS for complete coverage.
4. Keep DESNZ-CCC sector comparison cautious unless the sector mapping is strengthened.